# Merged `Saliency` verifying it returns equal values to the non-merged Saliency

One `Saliency` class, three model types.

Saliency is the only gradient method with no perturbation of its own, so the merged class
instantiates the modality's base tensor perturbator directly instead of mixing a method half
into it. That makes this the narrowest of the merges: the only thing that can differ from the
legacy classes is the plumbing, not the maths.

The last cell checks equality.

## Disable the runtime type checks (optional)

**Must run before anything imports `jaxtyping`** - the flag is read at import time, so this
has to be the first cell executed in a fresh kernel. Comment it out to get the shape checks
back; do that before believing any actual attribution numbers, since with the checks off a
shape bug surfaces as a downstream broadcast error or not at all.

In [ ]:
%matplotlib inline

import torch
from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoModelForCausalLM,
    AutoModelForImageClassification,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

from interpreto.attributions.methods.old_image_saliency import ImageSaliency as LegacyImageSaliency
from interpreto.attributions.methods.old_saliency import Saliency as LegacySaliency
from interpreto.attributions.methods.saliency import Saliency
from interpreto.visualizations import plot_attributions, plot_image_attribution

sentence = "interpreto is nice"
image = Image.open("../fixtures/images/equal_cat_and_dog.jpg").convert("RGB")
image

## The three models

In [ ]:
cls_model = AutoModelForSequenceClassification.from_pretrained("hf-internal-testing/tiny-random-bert")
cls_processor = AutoTokenizer.from_pretrained("hf-internal-testing/tiny-random-bert")

gen_model = AutoModelForCausalLM.from_pretrained("hf-internal-testing/tiny-random-gpt2")
gen_processor = AutoTokenizer.from_pretrained("hf-internal-testing/tiny-random-gpt2")

vit_model = AutoModelForImageClassification.from_pretrained("akahana/vit-base-cats-vs-dogs")
vit_processor = AutoImageProcessor.from_pretrained("akahana/vit-base-cats-vs-dogs")

## The three explainers

In [ ]:
cls_explainer = Saliency(
    cls_model,
    cls_processor,
)

gen_explainer = Saliency(
    gen_model,
    gen_processor,
)

vit_explainer = Saliency(
    vit_model,
    vit_processor,
)

legacy_cls_explainer = LegacySaliency(cls_model, cls_processor)
legacy_gen_explainer = LegacySaliency(gen_model, gen_processor)
legacy_vit_explainer = LegacyImageSaliency(vit_model, vit_processor)

# Saliency draws no random numbers, so the seeds are only there to keep this cell
# interchangeable with the other merge notebooks.
torch.manual_seed(0)
legacy_cls_outputs = legacy_cls_explainer.explain(sentence)
torch.manual_seed(0)
legacy_gen_outputs = legacy_gen_explainer.explain(sentence, targets="right")
torch.manual_seed(0)
legacy_vit_outputs = legacy_vit_explainer.explain(image)

torch.manual_seed(0)
cls_outputs = cls_explainer.explain(sentence)
torch.manual_seed(0)
gen_outputs = gen_explainer.explain(sentence, targets="right")
torch.manual_seed(0)
vit_outputs = vit_explainer.explain(image)

## Checking that the new methods are equal to the old

We consider here that LegacySaliency (on all task / modalities combination) is the ground truth.
We want to show that the merged Saliency method returns the same AttributionOutput /
ImageAttributionOutput as the ""Legacy""Saliency

In [ ]:
def same(a, b):
    """Exact equality, counting NaN in the same position as a match."""
    return a.shape == b.shape and bool(((a == b) | (a.isnan() & b.isnan())).all())


for name, new, old in [
    ("cls", cls_outputs[0], legacy_cls_outputs[0]),
    ("gen", gen_outputs[0], legacy_gen_outputs[0]),
    ("vit", vit_outputs[0], legacy_vit_outputs[0]),
]:
    ok = same(new.attributions, old.attributions)
    print(f"{name}: {tuple(new.attributions.shape)}  {'identical' if ok else 'DIFFERENT'}")
    assert ok, f"{name}: merged attributions differ from legacy"

print("\nmerged Saliency reproduces the legacy attributions exactly")

## Visualize

In [ ]:
plot_attributions(cls_outputs[0])

In [ ]:
plot_attributions(gen_outputs[0])

In [ ]:
plot_attributions(legacy_gen_outputs[0])

In [ ]:
fig, axes = plot_image_attribution(vit_outputs, alpha=0.5)